# Notebook 01 : Data Cleaning

**Goal:** Load raw Superstore data, fix issues, add derived columns, save clean version.

**Output:** `../data/superstore_clean.csv`

In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/superstore_raw.csv', encoding='latin-1')
print(f'Shape: {df.shape}')
df.head()

## Step 1 — Inspect

In [ ]:
print('Null counts:')
print(df.isnull().sum())
print('\nData types:')
print(df.dtypes)

## Step 2 — Fix data types

Convert date strings to datetime objects so we can do time-based analysis.

In [ ]:
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date'] = pd.to_datetime(df['Ship Date'])
df['Ship Days'] = (df['Ship Date'] - df['Order Date']).dt.days
print('Date columns converted.')
df[['Order Date', 'Ship Date', 'Ship Days']].head()

## Step 3 — Remove duplicates and critical nulls

Dropping nulls only in Sales and Profit — other nulls (like Postal Code) are acceptable.

In [ ]:
before = len(df)
df.drop_duplicates(inplace=True)
df.dropna(subset=['Sales', 'Profit'], inplace=True)
after = len(df)
print(f'Rows before: {before} | After: {after} | Removed: {before - after}')

## Step 4 — Add derived columns

These encode business concepts and will power most of the analysis.

In [ ]:
df['Profit Margin'] = df['Profit'] / df['Sales']
df['Revenue per Unit'] = df['Sales'] / df['Quantity']
df['Order Year'] = df['Order Date'].dt.year
df['Order Month'] = df['Order Date'].dt.month
df['Order Quarter'] = df['Order Date'].dt.quarter
df['Is Profitable'] = df['Profit'] > 0

print('Derived columns added.')
df[['Sales', 'Profit', 'Profit Margin', 'Is Profitable']].describe()

## Step 5 — Save clean dataset

In [ ]:
df.to_csv('../data/superstore_clean.csv', index=False)
print(f'Saved. Final shape: {df.shape}')

## Quick validation

Run these assertions to confirm everything looks right before moving to notebook 02.

In [ ]:
assert df.shape[0] > 9000, 'Row count too low'
assert 'Profit Margin' in df.columns, 'Derived column missing'
assert df['Sales'].isnull().sum() == 0, 'Nulls in Sales'
assert df['Profit'].isnull().sum() == 0, 'Nulls in Profit'
print('All validation checks passed.')